
# 청년 순이동 월간 경보 백테스트 (KOSIS 시군구 월별, 2005 ~ 2026.08)

**질문:** "우리 동네 청년(20~39세)이 예상보다 많이 빠져나가거나 들어오면 매달 알려드립니다"라는 서비스는 과거에 얼마나 정확했을까? 그리고 **지금(2026년) 어디에서 경보가 울리고 있을까?**

혼인 감시 실험에서는 시군구 월별 자료가 1년에 한 번만 공표돼 실시간 경보가 불가능했습니다. 인구이동 통계는 **시군구 × 연령 단위로 매달, 약 1개월 지연**으로 공표됩니다(2026-10-08 기준 2026.08까지). 그래서 실시간 경보를 실제로 운영할 수 있습니다.

**설계 (검증 방식 7단계, 채점 전에 고정)**
- **예측 (플랜 A, 모든 지역에 같은 방식):** 전입·전출 각각 "작년 연간 × 최근 5년 평균 월별 비중". 순이동 예측 = 전입 예측 − 전출 예측.
- **이탈률:** (실제 순이동 − 예측 순이동) ÷ 예측 전입, 1월부터 누적.
- **이탈 해와 정상 해:** 보정 기간(2010~2017) |이탈률| 분포의 상위 20%를 이탈 해, 하위 50%를 정상 해로 봅니다. 기준값은 보정 기간에서만 계산합니다.
- **평가 대상:** 직전 연도 청년 전입 2,000명 이상 시·군·자치구
- **경보:**
  - 순이동 잔차의 양방향 CUSUM에 실무 크기 조건(누적 이탈률 ≥ m)을 더합니다.
  - CUSUM 기준 h는 "보정 기간 정상 해의 잘못된 경보 10% 이하인 가장 민감한 값"으로 고릅니다.
  - m은 "가장 큰 규모 구간의 잘못된 경보 10% 이하인 가장 작은 값"으로 고릅니다.
- **채점:** 2018~2025년. 실시간 목록은 2026년 1~8월입니다.

자료: 통계청 KOSIS `DT_1B26001` (가공). 저장소 CSV를 쓰므로 키가 필요 없습니다.


In [ ]:
import os, numpy as np, pandas as pd

DATA = "data_kosis/youth_migration_monthly_sigungu_2005_2026.csv"
DATA_URL = ("https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/"
            "experiments/data_kosis/youth_migration_monthly_sigungu_2005_2026.csv")

# ---- 미리 정한 것 (채점 기간 결과를 보기 전에 고정) ----
CAL = range(2010, 2018)      # 기준을 정하는 기간 (예측 연도)
TEST = range(2018, 2026)     # 채점 기간
LIVE = 2026                  # 실시간 (1~8월 공표분)
MIN_SIZE = 2000              # 직전 연도 청년(20~39세) 전입 2,000명 이상 시군구만 평가
DEV_Q, CLEAR_Q = 0.80, 0.50  # 이탈 = 보정 기간 |이탈률| 상위 20%, 정상 = 하위 50% (기준값은 보정 기간에서만 계산)
K = 0.25                     # CUSUM 허용 폭
MAX_FA = 0.10                # 보정 기간 정상 해의 잘못된 경보율 10% 이하가 되는 가장 민감한 기준
LAG = 1                      # 월별 인구이동 공표 지연 (다음 달 말 공표 → 약 1개월)

def load(path=DATA):
    d = pd.read_csv(path if os.path.exists(path) else DATA_URL, dtype={"code": str, "ym": str})
    d["year"] = d.ym.str[:4].astype(int); d["month"] = d.ym.str[4:].astype(int)
    return d

def tables(d):
    """지역 × (연, 월) 전입·전출 표 (20~39세 합계)."""
    wide = lambda col: d.pivot_table(index="code", columns=["year", "month"], values=col)
    return wide("in_"), wide("out")

def forecasts(IN, OUT, years_needed=6):
    """플랜 A: 전입·전출 각각 '작년 연간 × 최근 5년 평균 월별 비중'. 순이동 예측 = 전입 예측 − 전출 예측.
    {(지역, 연도): dict(mu_in, mu_out, a_in, a_out, prev_in, n_months)}"""
    out = {}
    years = sorted({y for y, _ in IN.columns})
    def plan_a(row, Y):
        M = np.array([[row.get((y, m), np.nan) for m in range(1, 13)] for y in [Y - i for i in range(1, 6)]])
        if np.isnan(M).any(): return None, None
        shares = (M / M.sum(1, keepdims=True)).mean(0)
        return M[0].sum() * shares, M[0].sum()
    for code in IN.index:
        if code not in OUT.index: continue
        ri, ro = IN.loc[code], OUT.loc[code]
        for Y in years:
            if any(Y - i not in years for i in range(1, 6)): continue
            mi, prev_in = plan_a(ri, Y); mo, _ = plan_a(ro, Y)
            if mi is None or mo is None: continue
            ai = np.array([ri.get((Y, m), np.nan) for m in range(1, 13)])
            ao = np.array([ro.get((Y, m), np.nan) for m in range(1, 13)])
            n = int(np.sum(~np.isnan(ai) & ~np.isnan(ao)))
            if n == 0: continue
            out[(code, Y)] = dict(mu_in=mi, mu_out=mo, a_in=ai, a_out=ao, prev_in=prev_in, n=n)
    return out

def dev_rate(f, upto=12):
    """이탈률 = (실제 순이동 − 예측 순이동) 누적 / 예측 전입 누적."""
    s = slice(0, upto)
    return (np.nansum(f["a_in"][s] - f["a_out"][s]) - np.sum(f["mu_in"][s] - f["mu_out"][s])) / np.sum(f["mu_in"][s])

def phi_from(F, years):
    r = []
    for (c, y), f in F.items():
        if y in years and f["prev_in"] >= MIN_SIZE and f["n"] == 12:
            v = f["mu_in"] + f["mu_out"]
            r.append((((f["a_in"] - f["a_out"]) - (f["mu_in"] - f["mu_out"])) ** 2 / v).mean())
    return float(np.mean(r))

def thresholds(F):
    x = np.abs([dev_rate(f) for (c, y), f in F.items() if y in CAL and f["prev_in"] >= MIN_SIZE and f["n"] == 12])
    return float(np.quantile(x, DEV_Q)), float(np.quantile(x, CLEAR_Q))

def alarm(f, phi, h, m_min=0.0, k=K):
    """순이동 잔차의 양방향 CUSUM + (선택) 누적 이탈률 ≥ m_min. 첫 경보 월과 방향(+1 순유입 증가 / −1 순유출 증가)."""
    z = ((f["a_in"] - f["a_out"]) - (f["mu_in"] - f["mu_out"])) / np.sqrt(phi * (f["mu_in"] + f["mu_out"]))
    cp = cm = 0.0
    for m in range(f["n"]):
        cp = max(0.0, cp + z[m] - k); cm = max(0.0, cm - z[m] - k)
        r = dev_rate(f, m + 1)
        if cp > h and r >= m_min: return m + 1, 1
        if cm > h and r <= -m_min: return m + 1, -1
    return None, 0

def evaluate(F, years, phi, h, dev, clear, m_min=0.0):
    rows = []
    for (c, y), f in F.items():
        if y not in years or f["prev_in"] < MIN_SIZE or f["n"] < 12: continue
        m, s = alarm(f, phi, h, m_min)
        rows.append(dict(code=c, year=y, dev=dev_rate(f), alarm_m=m, alarm_dir=s, prev_in=f["prev_in"]))
    R = pd.DataFrame(rows)
    pos, neg = R[R.dev.abs() >= dev], R[R.dev.abs() < clear]
    hit = pos[pos.alarm_dir == np.sign(pos.dev)]
    return R, dict(지역연도=len(R), 이탈해=len(pos), 정상해=len(neg),
                   감지율=round(len(hit) / max(len(pos), 1), 3),
                   상반기내_감지율=round(float((hit.alarm_m <= 6).sum()) / max(len(pos), 1), 3),
                   감지월_중앙값=float(hit.alarm_m.median()) if len(hit) else np.nan,
                   잘못된경보율=round(float((neg.alarm_dir != 0).mean()), 3) if len(neg) else np.nan)

def by_size(R, dev, clear, cuts=(0, 5000, 15000, 1e9), labels=("2천~5천", "5천~1.5만", "1.5만 이상")):
    R = R.copy()
    R["규모"] = pd.cut(R.prev_in, list(cuts), labels=list(labels))
    R["이탈"] = R.dev.abs() >= dev; R["정상"] = R.dev.abs() < clear
    R["감지"] = R.이탈 & (R.alarm_dir == np.sign(R.dev))
    def s(x):
        return pd.Series(dict(지역연도=len(x), 이탈해=int(x.이탈.sum()), 감지율=round(x.감지.sum() / max(x.이탈.sum(), 1), 2),
                              감지월_중앙값=x[x.감지].alarm_m.median(), 정상해=int(x.정상.sum()),
                              잘못된경보율=round(float((x[x.정상].alarm_dir != 0).mean()), 2)))
    return pd.concat([R.groupby("규모", observed=True).apply(s), s(R).to_frame("전체").T])


## 1. 기준 정하기(보정 기간)와 채점(2018~2025)

In [ ]:
d = load()
names = d.drop_duplicates("code").set_index("code").name
IN, OUT = tables(d)
F = forecasts(IN, OUT)
phi = phi_from(F, CAL)
DEV, CLEAR = thresholds(F)
H_GRID = [2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 10, 12]
for h in H_GRID:                                   # 보정 기간: 정상 해 잘못된 경보율 ≤ 10%인 가장 민감한 h
    _, cal = evaluate(F, CAL, phi, h, DEV, CLEAR)
    if cal["잘못된경보율"] <= MAX_FA: break
M_GRID = [0, 0.01, 0.02, 0.03, 0.05, 0.075, 0.1]
for m_cal in M_GRID:                               # 보정 기간: 가장 큰 규모 구간의 잘못된 경보율 ≤ 10%인 가장 작은 실무 크기
    Rc, _ = evaluate(F, CAL, phi, h, DEV, CLEAR, m_cal)
    if by_size(Rc, DEV, CLEAR).iloc[-2]["잘못된경보율"] <= MAX_FA: break
print(f"시군구 {IN.shape[0]}곳 · 과분산 φ {phi:.2f} · 이탈 기준 |이탈률| ≥ {DEV:.1%}, 정상 < {CLEAR:.1%} (보정 기간 분포)")
print(f"보정으로 고른 값: CUSUM h = {h}, 실무 크기 m = {m_cal:.0%}")
R0, t0 = evaluate(F, TEST, phi, h, DEV, CLEAR)
R1, t1 = evaluate(F, TEST, phi, h, DEV, CLEAR, m_cal)
pd.DataFrame({"CUSUM만": t0, f"CUSUM + 누적 ±{m_cal:.0%}": t1}).T


In [ ]:

# 규모별 (위: CUSUM만, 아래: + 실무 크기 조건)
pd.concat({"CUSUM만": by_size(R0, DEV, CLEAR), f"+ 누적 ±{m_cal:.0%}": by_size(R1, DEV, CLEAR)})


## 2. 지금 울리고 있는 경보 (2026년 1~8월 공표분)

In [ ]:
SIDO = {"11": "서울", "12": "전남광주", "26": "부산", "27": "대구", "28": "인천", "29": "광주", "30": "대전", "31": "울산",
        "36": "세종", "41": "경기", "43": "충북", "44": "충남", "46": "전남", "47": "경북", "48": "경남", "50": "제주",
        "51": "강원", "52": "전북"}
label = lambda c: f"{SIDO.get(c[:2], c[:2])} {names[c].replace(' ', '')}"

def live_alarms(F, year=LIVE):
    """올해 공표분까지의 실시간 경보 목록 (보정 기간에서 고른 h·m 그대로).
    상태: 지금도 같은 방향으로 실무 크기 이상 벗어나 있으면 '유지', 아니면 '해소'."""
    rows = []
    for (c, y), f in F.items():
        if y != year or f["prev_in"] < MIN_SIZE: continue
        m, s = alarm(f, phi, h, m_cal)
        if s == 0: continue
        n = f["n"]; r = dev_rate(f, n)
        net = np.nansum(f["a_in"][:n] - f["a_out"][:n]); net_hat = np.sum(f["mu_in"][:n] - f["mu_out"][:n])
        rows.append(dict(지역=label(c), 반영=f"1~{n}월", 경보월=f"{m}월", 방향="순유입↑" if s > 0 else "순유출↑",
                         실제=int(net), 예측=int(round(net_hat)), 차이=int(round(net - net_hat)), 이탈률=f"{r:+.1%}",
                         상태="유지" if np.sign(r) == s and abs(r) >= m_cal else "해소"))
    return pd.DataFrame(rows).sort_values(["상태", "차이"], key=lambda s: s if s.name == "상태" else -s.abs()).reset_index(drop=True)

live = live_alarms(F)
print(f"2026년 평가 대상 {sum(1 for (c, y), f in F.items() if y == LIVE and f['prev_in'] >= MIN_SIZE)}곳 중 경보 {len(live)}곳")
live

In [ ]:

import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"); plt.rcParams["font.family"] = "NanumGothic"
except Exception:
    pass

def show(code, year=LIVE, ax=None):
    f = F[(code, year)]; n = f["n"]
    ax = ax or plt.gca()
    ax.plot(range(1, 13), np.cumsum(f["mu_in"] - f["mu_out"]), "o--", label="예측 누적 순이동")
    ax.plot(range(1, n + 1), np.nancumsum((f["a_in"] - f["a_out"])[:n]), "o-", label="실제 누적 순이동")
    m, s = alarm(f, phi, h, m_cal)
    if m: ax.axvline(m, color="r", ls=":", label=f"경보 {m}월")
    ax.axhline(0, color="k", lw=0.8); ax.set_title(f"{label(code)} {year} 청년(20~39세)"); ax.set_xlabel("월"); ax.legend(fontsize=8)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
for a, c in zip(ax, ["11740", "41170", "27200"]):   # 서울 강동구, 경기 안양시, 대구 남구
    show(c, ax=a)
plt.tight_layout(); plt.show()



## 3. 사전 실행 결과와 해석

**경보 성능 (채점 기간 2018~2025, 기준은 2010~2017에서만 정함)**

보정으로 고른 값: CUSUM h = 3.5, 실무 크기 m = ±5%. 이탈 해 기준은 |이탈률| ≥ 9.8%, 정상 해 기준은 < 4.4%입니다.

| 경보 | 이탈 해 | 감지율 | 상반기 내 | 감지 월 중앙값 | 잘못된 경보율 |
|---|---|---|---|---|---|
| CUSUM만 | 342 | 85% | 51% | 6월 | 12.9% |
| **CUSUM + 누적 ±5%** | 342 | **85%** | **51%** | **6월** | **8.2%** |

| 직전 연도 청년 전입 | 이탈 해 | 감지율 | 잘못된 경보율 (CUSUM만 → +실무 조건) |
|---|---|---|---|
| 2천~5천 | 100 | 57% | 2% → 1% |
| 5천~1.5만 | 127 | 93% | 10% → 9% |
| 1.5만 이상 | 115 | 99% | 18% → **10%** |

- **감지율 85%, 절반은 6월까지 잡습니다.** 공표 지연(약 1개월)을 더해도 연말 연간 통계보다 반년 빠릅니다.
- **CUSUM만 쓰면 채점 기간 잘못된 경보가 12.9%로 목표(10%)를 넘었습니다.** 실무 크기 조건을 더하면 8.2%로 목표 안에 들어옵니다. 혼인 실험과 같은 결론입니다.
- 작은 지역(청년 전입 2천~5천)은 잡음이 커서 감지율이 57%로 낮습니다.

**2026년 실시간 경보 (1~8월 공표분):** 평가 대상 159곳 중 31곳에서 경보가 울렸고, 30곳은 지금도 유지 중입니다. 광명시는 1월 경보 뒤 해소됐습니다.

| 예시 | 무엇이 일어났나 | 의미 |
|---|---|---|
| 서울 강동구 (2월 경보, −5,615명) | 청년 순유입이 2023년 1,396명 → 2024년 7,287명 → 2025년 7,705명으로 급증(2024년 12월부터)했다가 올해 사라짐 | 급증 시기가 대단지 입주 시기와 겹칩니다(원인은 추정). "작년 값" 예측이 일회성 급증을 반복한다고 가정해 생긴 이탈입니다. **입주 일정 같은 휘슬이 있으면 미리 예측에 반영**할 수 있었던 사례입니다 |
| 대구 남구 (+2,898명), 부산 남구 (+3,142명) | 예상 밖의 청년 순유입 | 해당 지자체·상권에는 반가운 신호. 원인 확인 필요 |
| 경기 안양시 (−2,317명), 양주시 (−1,829명) | 예상보다 청년 순유출 증가 | 인구 정책·상권에 경고 |

### 알게 된 것
1. **혼인과 달리 이 상품은 지금 바로 운영할 수 있습니다.** 공개 자료가 매달 시군구 단위로 나옵니다.
2. **"통계 조건 + 실무 크기 조건" 경보 규칙이 두 번째 자료에서도 재현됐습니다.** 혼인과 인구이동 모두에서 같은 규칙이 작동했습니다.
3. **이탈의 상당수는 아파트 입주처럼 미리 알 수 있는 사건 때문입니다.** 입주 예정 물량 같은 휘슬 자료를 붙이면 예측과 경보가 함께 좋아질 것으로 보입니다(아직 검증하지 않음).

### 한계
- 이탈 해와 정상 해 기준이 보정 기간 분포의 분위수라서, 절대 크기(명)가 아니라 상대 크기입니다.
- 실시간 목록은 원인을 말해 주지 않습니다. 원인은 휘슬 자료나 현지 확인이 필요합니다.
- 2026년 7월 행정구역 개편 지역은 옛 코드 기준 1~6월(또는 7월)분만 반영했습니다.
